In [40]:
import glob

import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast
from utils.hist_utils import get_highest_channels, fit_and_scale_mc

ROOT.EnableImplicitMT()
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.Declare('#include "const.h"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

print(channName)

%jsroot

{0: 'Data', 1: 'Signal', 2: 'Regeneration', 3: 'Omega', 4: '3pi0', 5: 'Semileptonic', 6: 'Other', 7: 'pi+pi-pi+pi-', 8: 'MC sum'}


In [ ]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/hal/ALL_PHYS_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS2_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS3_SEMILEPTONIC_NoSmearing/*.root"
)

data_files = []

for i in range(1, 46, 1):
    data_files += glob.glob(
        f"/workspace/root_files/hal/DATA_SEMILEPTONIC_NoSmearing/dk0*_{i}.root"
    )

# Użycie (błyskawiczne!):
all_files = get_valid_files_fast(all_files)
data_files = get_valid_files_fast(data_files)

# Scaling factor
lumi_data = 817216.9875
lumi_mc = 745800.1127
scaling_factor = 45 / 135
print(f"Lumi factor: {scaling_factor:.4f}")

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files)
df_data = ROOT.RDataFrame("h1", data_files)

mPi0 = 134.9768 # MeV
mPiCh = 139.57039 # MeV
mK0 = 497.611 # MeV

# --- PRZETWARZANIE DANYCH (DATA) ---
df_data = (
    df_data.Define("Kchrec_5", f"Kchrec[5] - {mK0}")
    .Define("Knerec_5", f"Knerec[5] - {mK0}")
    .Define("KnerecSix_5", f"KnerecSix[5] - {mK0}")
    .Define("KnetriKinFit_5", f"KnetriKinFit[5] - {mK0}")
    .Define("MassPi01", f"pi01[5] - {mPi0}")
    .Define("MassPi02", f"pi02[5] - {mPi0}")
    .Define("MassPi01Fit", f"pi01Fit[5] - {mPi0}")
    .Define("MassPi02Fit", f"pi02Fit[5] - {mPi0}")
    .Define("MassPi0NeutralFit1", f"pi0NeutralFit1[5] - {mPi0}")
    .Define("MassPi0NeutralFit2", f"pi0NeutralFit2[5] - {mPi0}")
    .Define("MassPi0PMFit1", f"pi0PMFit1[5] - {mPi0}")
    .Define("MassPi0PMFit2", f"pi0PMFit2[5] - {mPi0}")
    .Define("rho_ch", "calc_cylindrical_radius(Kchrec, ip)")
    .Define("rho_ne", "calc_cylindrical_radius(Knerec, ip)")
    .Define("z_ch", "calc_z_component(Kchrec, ip)")
    .Define("z_ne", "calc_z_component(Knerec, ip)")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecClosest[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(Knerec, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecClosest, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "Knerec[8] - KchrecClosest[8]")
    .Define("z_pm_closest", "KchrecClosest[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_pm_00", "Kchrec[8] - Knerec[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1, Kchrec)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2, Kchrec)")
    ###############################################################################################    
    .Filter("mcflag == 0 && mctruth != -1")
)

# --- PRZETWARZANIE MONTE CARLO (MC) ---
df = (
    df.Define("Kchrec_5", f"Kchrec[5] - {mK0}")
    .Define("Knerec_5", f"Knerec[5] - {mK0}")
    .Define("KnerecSix_5", f"KnerecSix[5] - {mK0}")
    .Define("KnetriKinFit_5", f"KnetriKinFit[5] - {mK0}")
    .Define("MassPi01", f"pi01[5] - {mPi0}")
    .Define("MassPi02", f"pi02[5] - {mPi0}")
    .Define("MassPi01Fit", f"pi01Fit[5] - {mPi0}")
    .Define("MassPi02Fit", f"pi02Fit[5] - {mPi0}")
    .Define("MassPi0NeutralFit1", f"pi0NeutralFit1[5] - {mPi0}")
    .Define("MassPi0NeutralFit2", f"pi0NeutralFit2[5] - {mPi0}")
    .Define("MassPi0PMFit1", f"pi0PMFit1[5] - {mPi0}")
    .Define("MassPi0PMFit2", f"pi0PMFit2[5] - {mPi0}")
    .Define("rho_ch", "calc_cylindrical_radius(Kchrec, ip)")
    .Define("rho_ne", "calc_cylindrical_radius(Knerec, ip)")
    .Define("z_ch", "calc_z_component(Kchrec, ip)")
    .Define("z_ne", "calc_z_component(Knerec, ip)")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecClosest[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(Knerec, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecClosest, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "Knerec[8] - KchrecClosest[8]")
    .Define("z_pm_closest", "KchrecClosest[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_pm_00", "Kchrec[8] - Knerec[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1, Kchrec)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2, Kchrec)")
    ###############################################################################################    
    .Filter("mcflag == 1 && mctruth != -1")
)

# Filtrowanie dla próbki semileptonicznej (dziedziczy wszystkie nowe kolumny m_*_hyp)
df_Semileptonic = df.Filter(
    "mcflag == 1 && (mctruth == 5 || mctruth == 0 || mctruth == -1)"
)

# df Signal
df_Signal = df.Filter(
    f"mcflag == 1 && mctruth == 1"
)

df_Background = df.Filter(
    f"mcflag == 1 && mctruth != 1 && mctruth != -1"
)

In [ ]:
stats_semileptonic = {"counts": [], "efficiency": []}
counts_error_code = {300: 0, 301: 0, 303: 0, 307: 0}

efficiency_error_code = {code: 0 for code in counts_error_code.keys()}
relative_efficiency_error_code = {code: 0 for code in counts_error_code.keys()}

# General counts
stats_semileptonic["counts"].append(df_Semileptonic.Count())
stats_semileptonic["counts"].append(df_Semileptonic.Filter("mctruth == 0 || mctruth == 5").Count())
stats_semileptonic["counts"].append(df_Semileptonic.Filter("mctruth == 5").Count())

# Counts after each error code appearance
last_codes = ""
for code in counts_error_code.keys():
    filter = f"errorcode != {code} && !({last_codes})" if len(last_codes) > 0 else f"errorcode != {code}"
    
    counts_error_code[code] = df_Semileptonic.Filter(filter).Count()
    last_codes += f"|| errorcode == {code}" if len(last_codes) > 0 else f"errorcode == {code}"

ROOT.RDF.RunGraphs(stats_semileptonic["counts"])
ROOT.RDF.RunGraphs(list(counts_error_code.values()))

stats_semileptonic["efficiency"].append(stats_semileptonic["counts"][1].GetValue() / stats_semileptonic["counts"][0].GetValue())
stats_semileptonic["efficiency"].append(stats_semileptonic["counts"][2].GetValue() / stats_semileptonic["counts"][1].GetValue())

last_code = None
for code in counts_error_code.keys():
    efficiency_error_code[code] = counts_error_code[code].GetValue() / stats_semileptonic["counts"][0].GetValue()
    relative_efficiency_error_code[code] = efficiency_error_code[code] / efficiency_error_code[last_code] if last_code in efficiency_error_code else 0
    last_code = code

In [ ]:
print(f"Counts Semi Total: {stats_semileptonic['counts'][0].GetValue()} (100.00%)")
print(f"Counts Semi After Preselection: {stats_semileptonic['counts'][1].GetValue()} ({stats_semileptonic['efficiency'][0] * 100:.2f}%)")
print(f"Counts Semi After Cuts: {stats_semileptonic['counts'][2].GetValue()} ({stats_semileptonic['efficiency'][1] * 100:.2f}%)")

for code in counts_error_code.keys():
    print(f"Counts Semi After Error Code {code}: {counts_error_code[code].GetValue()} ({efficiency_error_code[code] * 100:.2f}%, Relative: {relative_efficiency_error_code[code] * 100:.2f}%)")

In [ ]:
# Generate overlapped histograms for all channels
variables_1D = ["Knerec_5", 
                "KnerecSix_5", 
                "Kchrec_5"]
variables_1D_titles = ["m^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]", 
                       "m^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]", 
                       "m^{inv}_{+-} - m_{K^{0}} [MeV/c^{2}]"]
range_X_1D = [(100, -400, 200), 
              (100, -400, 200), 
              (100, -200, 200)]

In [ ]:
# 1D histograms for all_channels
hist_1D = {}
modes = {"full": [], "zoom": []}

# Define histogram ranges for different modes
modes["full"] = [(100, -400, 200), 
                 (100, -400, 200), 
                 (100, -200, 200)]
modes["zoom"] = [(100, -100, 100), 
                (100, -100, 100), 
                (100, -5, 5)]

for i, var in enumerate(variables_1D):
  hist_1D[var] = {}
  for mctruth, chann in channName.items():
    if chann == "MC sum":
          continue

    hist_1D[var][chann] = {}
    df_draw = df_data if chann == "Data" else df
    
    title_str = f";{variables_1D_titles[i]};Counts"
    for mode in modes:
      hist_1D[var][chann][mode] = df_draw.Filter(f"mctruth == {mctruth}").Histo1D(
          (f"{var}_{chann}_{mode}", title_str, *modes[mode][i]), var
      )

ROOT.RDF.RunGraphs(list(hist_1D[var][chann][mode] for var in variables_1D for chann in channName.values() if chann != "MC sum" for mode in modes))

#### Create MC sum ####
for i, var in enumerate(variables_1D):
  hist_1D[var]["MC sum"] = {}
  for mode in modes:
    hist_1D[var][channName[1]][mode].Scale(0.3333)
    hist_1D[var]["MC sum"][mode] = hist_1D[var][channName[1]][mode].Clone()  # Initialize with the first MC channel
    for chann in list(channName.values())[2:]:
        if chann != "MC sum" and chann != "Data":
          hist_1D[var][chann][mode].Scale(0.3333)
          hist_1D[var]["MC sum"][mode].Add(hist_1D[var][chann][mode].GetPtr())
######################

In [50]:
ROOT.TGaxis.SetMaxDigits(4)
ROOT.gStyle.SetOptStat(0)

canvas = []
mode_list = list(modes.keys())

### Get Max Channel per mode per var for scaling purposes
max_per_mode = {var: {mode: 0 for mode in mode_list} for var in variables_1D}
for var in variables_1D:
    for mode in mode_list:
        for chann in channName.values():
          h_curr = hist_1D[var][chann][mode]
          max_per_mode[var][mode] = max(max_per_mode[var][mode], h_curr.GetMaximum())

          print(f"var: {var}, mode: {mode}, chann: {chann}, max: {max_per_mode[var][mode]}")

for i, var in enumerate(variables_1D):
    canvas.append(ROOT.TCanvas(f"c_{i}", f"c_{i}", 800, 400))
    canvas[-1].Divide(len(mode_list), 1)

    for j, mode in enumerate(mode_list):
        for k, chann in enumerate(channName.values()):
          canvas[-1].cd(j + 1)

          h_curr = hist_1D[var][chann][mode]
          h_curr.SetLineColor(channColor[chann])
          h_curr.SetMaximum(max_per_mode[var][mode] * 1.8)

          mc_data_draw_style = "PE1" if chann == "Data" else "HIST"

          if k == 0:
              h_curr.Draw(mc_data_draw_style)
          else:
              h_curr.Draw(f"{mc_data_draw_style} SAME")

    canvas[-1].Draw()
    canvas[-1].SaveAs(f"canvas_{i}.pdf")

var: Knerec_5, mode: full, chann: Data, max: 1270455.1800382405
var: Knerec_5, mode: full, chann: Signal, max: 1270455.1800382405
var: Knerec_5, mode: full, chann: Regeneration, max: 1270455.1800382405
var: Knerec_5, mode: full, chann: Omega, max: 1270455.1800382405
var: Knerec_5, mode: full, chann: 3pi0, max: 1270455.1800382405
var: Knerec_5, mode: full, chann: Semileptonic, max: 1270455.1800382405
var: Knerec_5, mode: full, chann: Other, max: 1270455.1800382405
var: Knerec_5, mode: full, chann: pi+pi-pi+pi-, max: 1270455.1800382405
var: Knerec_5, mode: full, chann: MC sum, max: 1270455.1800382405
var: Knerec_5, mode: zoom, chann: Data, max: 406196.1071323721
var: Knerec_5, mode: zoom, chann: Signal, max: 406196.1071323721
var: Knerec_5, mode: zoom, chann: Regeneration, max: 406196.1071323721
var: Knerec_5, mode: zoom, chann: Omega, max: 406196.1071323721
var: Knerec_5, mode: zoom, chann: 3pi0, max: 406196.1071323721
var: Knerec_5, mode: zoom, chann: Semileptonic, max: 406196.10713237

In [ ]:
variables_2D = [["Knerec_5", "KnerecSix_5"], 
                ["Chi2SignalKinFit", "Chi2NeutralKinFit"], 
                ["Chi2SignalKinFit", "Chi2PMKinFit"],
                ["MassPi01Fit", "MassPi02Fit"]]
variables_2D_titles = [["m^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]", "m^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]"], 
                       ["#chi^{2}_{+-00} [-]", "#chi^{2}_{00} [-]"], 
                       ["#chi^{2}_{+-00} [-]", "#chi^{2}_{+-} [-]"],
                       ["m^{inv,fit}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv,fit}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"]]
range_X_2D = [[(100, -400, 200), (100, -400, 200)], 
              [(100, 0, 100), (100, 0, 100)], 
              [(100, 0, 100), (100, 0, 30)], 
              [(100, -200, 300), (100, -200, 300)]]

In [ ]:
## 2D histograms for mctruth == 1
hist_2D = {}
modes = {}

# Definicja zakresów dla trybów: (nbinsX, xlow, xhigh, nbinsY, ylow, yhigh)
modes["full"] = [(100, -400, 200, 100, -400, 200), 
                 (100, 0, 100, 100, 0, 100), 
                 (100, 0, 100, 100, 0, 30),
                 (100, -200, 300, 100, -200, 300)]
modes["zoom"] = [(100, -100, 100, 100, -100, 100), 
                 (100, 0, 50, 100, 0, 50), 
                 (100, 0, 50, 100, 0, 15),
                 (100, -50, 50, 100, -50, 50)]

for idx, varset in enumerate(variables_2D):
    var_key = "_".join(varset)
    hist_2D[var_key] = {}
    title_str = f";{variables_2D_titles[idx][0]};{variables_2D_titles[idx][1]}"
    
    for mode, model_args in modes.items():
        hist_2D[var_key][mode] = df_Signal.Histo2D(
            (f"{var_key}_{mode}", title_str, *model_args[idx]),
            varset[0], varset[1]  # Przekazujemy obie zmienne dla 2D
        )

# Uruchomienie obliczeń dla wszystkich histogramów 2D
all_hists = [hist_2D[var_key][mode] for var_key in hist_2D for mode in modes]
ROOT.RDF.RunGraphs(all_hists)

In [ ]:
ROOT.TGaxis.SetMaxDigits(6)

canva = []

for i, var_key in enumerate(hist_2D):
    canva.append(ROOT.TCanvas(f"c2d_{i}", "2D Histograms", 800, 400))
    canva[-1].Divide(len(modes), 1)
    for j, mode in enumerate(modes):
      canva[-1].cd(j + 1)
      hist_2D[var_key][mode].Draw("COLZ")
    canva[-1].Draw()
    

### Check of signal cuts for a designed boundaries

In [ ]:
########### Fiducial volume Omega-pi0 #########################
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
###############################################################
########### Fiducial volume tracks back to back ###############
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"
###############################################################

cut1 = 8.086
cut2 = 11.515

cut_list = ["abs(Kchrec_5) <= 3 * 0.8090",
            "ellipse_cut_func(Knerec_5, KnerecSix_5)",
            f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
            f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
            f"m1_vs_m2_cut(MassPi01Fit, MassPi02Fit, {cut1}, {cut2})",
            "Chi2SignalKinFit < 30.0"
            ]

In [ ]:
signal_cut_stats = {}
background_cut_stats = {}

total_background_per_channel = {}

total_signal = df_Signal.Count()
total_background = df_Background.Count()

for mctruth, chann in channName.items():
    total_background_per_channel[chann] = df_Background.Filter(f"mctruth == {mctruth}").Count()

total_cut = ""
for cut in cut_list:
  if total_cut:
    total_cut += " && "
  total_cut += f"({cut})"

  signal_cut_stats[total_cut] = df_Signal.Filter(total_cut).Count()
  background_cut_stats[total_cut] = df_Background.Filter(total_cut).Count()

ROOT.RDF.RunGraphs(list(signal_cut_stats.values()) + [total_signal] + list(background_cut_stats.values()) + [total_background])

efficiencies = {cut: count.GetValue() / total_signal.GetValue() for cut, count in signal_cut_stats.items()}
background_efficiencies = {cut: count.GetValue() / total_background.GetValue() for cut, count in background_cut_stats.items()}
initial_purity = total_signal.GetValue() / (total_signal.GetValue() + total_background.GetValue())
purity = {cut: count.GetValue() / (count.GetValue() + background_cut_stats[cut].GetValue()) for cut, count in signal_cut_stats.items()}
relative_efficiencies = {}
previous_efficiency = 1.0
for cut, efficiency in efficiencies.items():
  relative_efficiencies[cut] = efficiency / previous_efficiency
  previous_efficiency = efficiency

from textwrap import fill

signal_total = total_signal.GetValue()
background_total = total_background.GetValue()

print("=" * 88)
print("SIGNAL CUT-FLOW SUMMARY")
print("-" * 88)
print(
    f"Initial sample | Signal: {signal_total:>12,} | "
    f"Background: {background_total:>12,} | Purity: {initial_purity:.2%}"
)

for index, (cut, count) in enumerate(signal_cut_stats.items(), start=1):
    print(f"\n{index:02d}. CUT")
    print(fill(cut, width=88, initial_indent="    ", subsequent_indent="    "))
    print(
        f"    Signal: {count.GetValue():>12,} | "
        f"Efficiency: {efficiencies[cut]:>7.2%} | "
        f"Relative: {relative_efficiencies[cut]:>7.2%}"
    )
    print(
        f"    Background efficiency: {background_efficiencies[cut]:>7.2%} | "
        f"Purity: {purity[cut]:>7.2%}"
    )

print("=" * 88)


### Test of cuts for separated kinematic fit

In [ ]:
########### Fiducial volume Omega-pi0 #########################
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
###############################################################
########### Fiducial volume tracks back to back ###############
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"
###############################################################

cut1 = 8.086
cut2 = 11.515

cut_list = ["abs(Kchrec_5) <= 3 * 0.8090",
            "ellipse_cut_func(Knerec_5, KnerecSix_5)",
            f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
            f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
            f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {cut1}, {cut2})",
            "Chi2NeutralKinFit < 30.0 && Chi2PMKinFit < 30.0"
            ]

In [54]:
signal_cut_stats = {}
background_cut_stats = {}

total_background_per_channel = {}

total_signal = df_Signal.Count()
total_background = df_Background.Count()

for mctruth, chann in channName.items():
    total_background_per_channel[chann] = df_Background.Filter(f"mctruth == {mctruth}").Count()

total_cut = ""
for cut in cut_list:
  if total_cut:
    total_cut += " && "
  total_cut += f"({cut})"

  signal_cut_stats[total_cut] = df_Signal.Filter(total_cut).Count()
  background_cut_stats[total_cut] = df_Background.Filter(total_cut).Count()

ROOT.RDF.RunGraphs(list(signal_cut_stats.values()) + [total_signal] + list(background_cut_stats.values()) + [total_background])

efficiencies = {cut: count.GetValue() / total_signal.GetValue() for cut, count in signal_cut_stats.items()}
background_efficiencies = {cut: count.GetValue() / total_background.GetValue() for cut, count in background_cut_stats.items()}
initial_purity = total_signal.GetValue() / (total_signal.GetValue() + total_background.GetValue())
purity = {cut: count.GetValue() / (count.GetValue() + background_cut_stats[cut].GetValue()) for cut, count in signal_cut_stats.items()}
relative_efficiencies = {}
previous_efficiency = 1.0
for cut, efficiency in efficiencies.items():
  relative_efficiencies[cut] = efficiency / previous_efficiency
  previous_efficiency = efficiency

from textwrap import fill

signal_total = total_signal.GetValue()
background_total = total_background.GetValue()

print("=" * 88)
print("SIGNAL CUT-FLOW SUMMARY")
print("-" * 88)
print(
    f"Initial sample | Signal: {signal_total:>12,} | "
    f"Background: {background_total:>12,} | Purity: {initial_purity:.2%}"
)

for index, (cut, count) in enumerate(signal_cut_stats.items(), start=1):
    print(f"\n{index:02d}. CUT")
    print(fill(cut, width=88, initial_indent="    ", subsequent_indent="    "))
    print(
        f"    Signal: {count.GetValue():>12,} | "
        f"Efficiency: {efficiencies[cut]:>7.2%} | "
        f"Relative: {relative_efficiencies[cut]:>7.2%}"
    )
    print(
        f"    Background efficiency: {background_efficiencies[cut]:>7.2%} | "
        f"Purity: {purity[cut]:>7.2%}"
    )

print("=" * 88)


SIGNAL CUT-FLOW SUMMARY
----------------------------------------------------------------------------------------
Initial sample | Signal:      164,399 | Background:   80,914,104 | Purity: 0.20%

01. CUT
    (abs(Kchrec_5) <= 3 * 0.8090)
    Signal:      114,440 | Efficiency:  69.61% | Relative:  69.61%
    Background efficiency:  35.22% | Purity:   0.40%

02. CUT
    (abs(Kchrec_5) <= 3 * 0.8090) && (ellipse_cut_func(Knerec_5, KnerecSix_5))
    Signal:      109,084 | Efficiency:  66.35% | Relative:  95.32%
    Background efficiency:  24.02% | Purity:   0.56%

03. CUT
    (abs(Kchrec_5) <= 3 * 0.8090) && (ellipse_cut_func(Knerec_5, KnerecSix_5)) &&
    ((rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 &&
    abs(z_00_closest) < 2.0 && rho_omega > 1.5) || !(rho_pm_closest < 2.0 &&
    rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0))
    Signal:      108,054 | Efficiency:  65.73% | Relative:  99.06%
    Background efficiency:  23.85% | P